# MorphoNup — SHAP and explainability

Explains the **saved** MorphoNup models (no training) on their own outer test rows; the background comes from each fold's training rows.

Feature attribution only (how much each input contributes); motif and sequence-pattern analyses are a separate study.

* **Group Shapley values** (exact, all 2^11 coalitions of sequence, k-mers, MGW, Roll, EP, PLS 1-6), read at three resolutions of the same game: input modality (Sequence, k-mers, Shape), shape tracks (PLS block as one), shape channels. At each level they sum to the prediction minus the background mean; levels are not additive across levels. Also pairwise interactions, results by prediction outcome, summary (beeswarm) and dependence plots.
* **Integrated gradients**: where along the 147 bp the sequence and each shape channel act, and a cross-check of the modality ranking against the Shapley values.
* **PLS loadings** to read the PLS players in terms of the 7 original shape tracks.
* Every rebuilt model is first checked against its saved test predictions.

These describe what the model uses; they are associations, not causal claims.

In [ ]:
DATASET = "DatasetNup_1/Homo_sapiens"
K_FOLDS = 5
SEED = 151000
FOLDS = None                 # None = every outer fold
N_TARGETS = 200              # explained test sequences per fold (half per class)
N_BACKGROUND = 50            # background sequences per fold (training rows)
STEPS = 32                   # integrated-gradients steps
RUN_MODE = "results_only"    # "background" | "inline" | "results_only"
ALLOW_BUSY_GPU = False

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
_null, _err = os.open(os.devnull, os.O_WRONLY), os.dup(2)
os.dup2(_null, 2)                                    # TensorFlow's CUDA start-up lines go to C-level stderr
try:
    import tensorflow  # noqa: F401
finally:
    os.dup2(_err, 2)
    os.close(_null)
    os.close(_err)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "morphonup" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display

from morphonup import config, figures, jobs
from morphonup.cv import run_dir

pd.set_option("display.width", 200, "display.max_columns", 40)


def show(path, width=900):
    """Display a saved figure with its caption (from the folder's figure_catalog.csv)."""
    path = Path(path)
    if not path.exists():
        print("missing:", path.relative_to(ROOT))
        return
    cat = path.parent / "figure_catalog.csv"
    caption = pd.read_csv(cat).set_index("png").suggested_caption.get(path.name, "") if cat.exists() else ""
    display(Markdown(f"**{path.stem}** — `{path.relative_to(ROOT)}`  \n{caption}"))
    display(Image(filename=str(path), width=width))
OUT = run_dir(DATASET, K_FOLDS) / "explain" / f"seed_{SEED}"
models = sorted(run_dir(DATASET, K_FOLDS).glob(f"seed_{SEED}/fold_*/refit/best.weights.h5"))
print(f"saved MorphoNup models for seed {SEED}: {len(models)} of {K_FOLDS} folds")
print("explained folds so far:", sorted(p.parent.name for p in OUT.glob("fold_*/done.json")) or "none")
gpu, procs = jobs.gpu_status()
print("GPU:", gpu, "| processes:", procs or "none")

## Run

In [ ]:
if RUN_MODE == "background":
    LOG = jobs.launch_explain(DATASET, K_FOLDS, SEED, FOLDS, N_TARGETS, N_BACKGROUND, STEPS,
                              allow_busy_gpu=ALLOW_BUSY_GPU)
    print("launched; log:", LOG.relative_to(ROOT))
elif RUN_MODE == "inline":
    gpu, procs = jobs.gpu_status()
    if procs and not ALLOW_BUSY_GPU:
        raise RuntimeError("GPU in use: " + "; ".join(procs))
    from morphonup.data import Dataset
    from morphonup.explain import explain
    explain(Dataset(config.dataset_folder(DATASET)), run_dir(DATASET, K_FOLDS), K_FOLDS, SEED, FOLDS,
            n_targets=N_TARGETS, n_background=N_BACKGROUND, steps=STEPS)
else:
    print("results_only: nothing started")
print(jobs.tail(jobs.log_path(DATASET, K_FOLDS, [SEED], kind="explain")))

## Results

In [ ]:
from morphonup.explain import explain_figures
explain_figures(OUT) if OUT.exists() else None
audit = OUT / "audit.json"
if audit.exists():
    import json
    folds = json.loads(audit.read_text())["folds"]
    display(pd.DataFrame(folds).T[["n_targets", "n_background", "rebuilt_vs_saved_max_abs_probability_difference",
                                   "shapley_additivity_max_error", "ig_completeness_median_abs_error",
                                   "seconds_shapley", "seconds_integrated_gradients"]])
    table = OUT / "shap_table.md"
    print(table.read_text() if table.exists() else "")
else:
    print("no explanation yet")

In [ ]:
if (OUT / "figures").exists():
    for name in ("shap_overview", "shap_modality", "shap_tracks", "shap_channels", "shap_beeswarm_tracks",
                 "shap_beeswarm_channels", "shap_dependence_tracks", "shap_distribution_tracks", "shap_by_outcome_tracks",
                 "shap_interactions_tracks", "attribution_agreement", "ig_sequence_profile", "ig_shape_profile",
                 "ig_position_heatmap", "pls_loadings"):
        show(OUT / "figures" / f"{name}.png")
else:
    print("No figures yet: run the explanation (RUN_MODE = 'background').")